# 决策树

In [11]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# 读取数据
df = pd.read_csv('data/titanic.csv')

# 选择特征
features = ['pclass', 'sex', 'age']

# 将所有列名转为小写，保证兼容性
df.columns = df.columns.str.lower()
df = df[features + ['survived']]
df = df.dropna(subset=features)  # 如果所有特征有任何缺失值都去除

# 将所有类别（str类型）特征转换为数值型
# pclass有可能是"1st"，"2nd"等字符串，要统一成数字
df['pclass'] = df['pclass'].replace({'1st': 1, '2nd': 2, '3rd': 3})
df['pclass'] = pd.to_numeric(df['pclass'], errors='coerce')

# 性别编码
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# 将包含仍为缺失值的行剔除
df = df.dropna(subset=features)

# 特征和标签
X = df[features].astype(float)
y = df['survived']

# 拆分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 决策树模型
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

# 模型准确率
acc = clf.score(X_test, y_test)
print('决策树模型测试集准确率: {:.2f}%'.format(acc * 100))

决策树模型测试集准确率: 83.46%


# 随机森林

In [12]:
from sklearn.ensemble import RandomForestClassifier

# 通过随机森林模型来缓解决策树可能存在的过拟合与不稳定等问题
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf_clf.fit(X_train, y_train)

rf_acc = rf_clf.score(X_test, y_test)
print('随机森林模型测试集准确率: {:.2f}%'.format(rf_acc * 100))

随机森林模型测试集准确率: 82.68%


# 网格搜索

In [13]:
from sklearn.model_selection import GridSearchCV

# 定义参数网格
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 建立GridSearchCV对象
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    scoring='accuracy'
)

# 执行网格搜索
grid_search.fit(X_train, y_train)

# 最佳参数和对应分数
print("最佳参数:", grid_search.best_params_)
print("训练集最佳得分: {:.2f}%".format(grid_search.best_score_ * 100))

# 用最佳参数在测试集上评估
best_rf = grid_search.best_estimator_
test_acc = best_rf.score(X_test, y_test)
print("测试集准确率（优化后）: {:.2f}%".format(test_acc * 100))

最佳参数: {'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
训练集最佳得分: 82.82%
测试集准确率（优化后）: 83.46%
